In [1]:
# ── Configuration (edit these paths) ──────────────────────────────────────
import os

# Path to the folder containing the media files (images and videos)
MEDIA_ROOT = r'../dataset/media'  # update if media is elsewhere

# Path to the CSV index file (created by 1_index_media.ipynb)
CSV_PATH = r'../dataset/labels/media_index.csv'


ModuleNotFoundError: No module named 'google.colab'

In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Image
from IPython.display import clear_output

In [ ]:
# Function to display an image
def show_image(img_path):
    with open(img_path, 'rb') as f:
        display(Image(data=f.read()))

In [ ]:
# Load CSV file with the specified delimiter
file_path = CSV_PATH
df = pd.read_csv(file_path, delimiter=';')

# Define the evaluator columns
evaluator_columns = {
    '1': ["Etici.1", "Non_Etici.1"],
    '2': ["Etici.2", "Non_Etici.2"],
    '3': ["Etici.3", "Non_Etici.3"]
}

# Ask the user which evaluator they are
evaluator_num = input("Inserisci il numero del valutatore (1, 2, 3): ")
evaluator_columns_selected = evaluator_columns[evaluator_num]

# Define the categories
categories = ["Etici", "Non_Etici"]

# Function to handle user input and update the dataframe
def handle_input(row):
    clear_output()
    print(f"Visualizza il contenuto per la riga {row['Collegamento']}")


    # Definisci le estensioni dei file che consideriamo immagini e video
    image_extensions = ['.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff']
    video_extensions = ['.mp4', '.mov', '.avi', '.mkv', '.flv', '.wmv']

    show_image("/content/drive/MyDrive/Video e Immagini/Dataset_Binario/"+row['Collegamento'].replace("\\","//"))

    # # Display the image or video
    # show_image(row['Collegamento'])
    # riproduci_video(row['Collegamento'])

    # Create checkboxes for each category
    checkboxes = {cat: widgets.Checkbox(value=False, description=cat) for cat in categories}
    checkbox_container = widgets.VBox(list(checkboxes.values()))

    # Function to handle the selection of 'Niente'
    def handle_niente(change):
        if change['new']:
            for cat, checkbox in checkboxes.items():
                if cat != "Niente":
                    checkbox.value = False
                    checkbox.disabled = True
        else:
            for cat, checkbox in checkboxes.items():
                checkbox.disabled = False

    # checkboxes['Niente'].observe(handle_niente, 'value')

    # Display checkboxes
    display(checkbox_container)

    # Button to submit the selection
    submit_button = widgets.Button(description="Submit")
    display(submit_button)

    # Button to terminate the evaluation
    terminate_button = widgets.Button(description="Terminate")
    display(terminate_button)

    # Button to skip the current row
    skip_button = widgets.Button(description="Skip")
    display(skip_button)

    # Function to handle button click for submission
    def on_button_click(b):
        selections = {cat: 1 if checkbox.value else 0 for cat, checkbox in checkboxes.items()}
        # if selections['Niente']:
        #     for cat in categories[:-1]:
        #         selections[cat] = 0
        for cat, val in selections.items():
            col_name = evaluator_columns_selected[categories.index(cat)]
            df.loc[df.index == row.name, col_name] = val
        next_row = df.loc[df.index == row.name + 1]
        if not next_row.empty:
            handle_input(next_row.iloc[0])
        else:
            print("Valutazione completata!")
            df.to_csv(file_path, index=False, sep=';')
            print("File CSV aggiornato e salvato.")

    # Function to handle button click for termination
    def on_terminate_click(b):
        df.to_csv(file_path, index=False, sep=';')
        print("Valutazione terminata anticipatamente.")
        print("File CSV aggiornato e salvato.")
        clear_output()

    # Function to handle button click for skipping a row
    def on_skip_click(b):
        next_row = df.loc[df.index == row.name + 1]
        if not next_row.empty:
            handle_input(next_row.iloc[0])
        else:
            print("Valutazione completata!")
            df.to_csv(file_path, index=False, sep=';')
            print("File CSV aggiornato e salvato.")

    submit_button.on_click(on_button_click)
    terminate_button.on_click(on_terminate_click)
    skip_button.on_click(on_skip_click)

# Start the evaluation process from the first row
handle_input(df.iloc[0])
